# Phase 0 recon map: harvest areas over chlorophyll-a

Weekly mean of IMOS VIIRS chlorophyll-a (NOAA-20 + SNPP, OCI algorithm, ~1 km) for the recon week,
with the Tasmanian ShellMAP harvest-area polygons (theLIST layer 92) on top.

* SA harvest polygons are **not available yet** (pir.sa.gov.au blocks automated requests, permission requested), so the SA panel shows chlorophyll only.
* Tasmanian polygons are for **local research only** until theLIST confirms the licence.
* Inputs come from `pipeline.fetch.recon_imos` and `pipeline.fetch.recon_tas_boundaries`.

In [1]:
import glob

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import LogNorm

from pipeline.config import RAW_DIR, REPO_ROOT
from pipeline.qc import mask_invalid_chl, valid_fraction

In [2]:
def weekly_chl(region: str) -> xr.DataArray:
    """Mean of all valid daily VIIRS chl-a pixels (both satellites) for one region."""
    files = sorted(glob.glob(str(RAW_DIR / "imos" / "oc_*_chl_oci" / region / "*.nc")))
    days = [mask_invalid_chl(xr.open_dataset(f).chl_oci.isel(time=0, drop=True)) for f in files]
    stack = xr.concat(days, dim="file")
    return stack.mean("file", skipna=True)


chl = {r: weekly_chl(r) for r in ["south_australia", "tasmania"]}
for r, da in chl.items():
    print(f"{r}: weekly composite valid fraction = {valid_fraction(da):.0%}")

tas_areas = gpd.read_file(RAW_DIR / "thelist" / "shellmap_harvest_areas.geojson")
print(len(tas_areas), "Tas harvest-area polygons;", tas_areas.GA_NAME.nunique(), "growing areas")
print("median area:", round(tas_areas.AREA_HA.median()), "ha  (one 1 km pixel = 100 ha)")

south_australia: weekly composite valid fraction = 58%
tasmania: weekly composite valid fraction = 66%
57 Tas harvest-area polygons; 29 growing areas
median area: 133 ha  (one 1 km pixel = 100 ha)


In [3]:
norm = LogNorm(vmin=0.1, vmax=10)
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, (region, da) in zip(axes, chl.items(), strict=True):
    im = da.plot.pcolormesh(
        ax=ax,
        x="longitude",
        y="latitude",
        norm=norm,
        cmap="viridis",
        add_colorbar=False,
        rasterized=True,
    )
    ax.set_title(region.replace("_", " ").title())
    ax.set_aspect(1 / np.cos(np.deg2rad(float(da.latitude.mean()))))
    ax.set_facecolor("0.85")  # grey = land or cloud (no valid pixel all week)
tas_areas.boundary.plot(ax=axes[1], color="red", linewidth=1.2)
axes[1].text(0.02, 0.02, "red = ShellMAP harvest areas", transform=axes[1].transAxes, color="red")
axes[0].text(0.02, 0.02, "SA polygons pending (PIRSA)", transform=axes[0].transAxes)
fig.colorbar(im, ax=axes, label="chlorophyll-a (mg m$^{-3}$), weekly mean 1-7 Nov 2026", shrink=0.8)
fig.suptitle("Phase 0: IMOS VIIRS chl-a with harvest areas. Grey = land or no valid pixel")
out = REPO_ROOT / "docs" / "img" / "phase0_recon_map.png"
fig.savefig(out, dpi=110, bbox_inches="tight")
print("saved", out)

saved C:\Users\timne\Desktop\AlgalBloom\docs\img\phase0_recon_map.png


In [4]:
# Zoom: how many satellite pixels actually fall inside / near each Tasmanian harvest area?
tas = chl["tasmania"]
lon2d, lat2d = np.meshgrid(tas.longitude.values, tas.latitude.values)
pix = gpd.GeoDataFrame(
    {"chl": tas.values.ravel()},
    geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()),
    crs=4326,
).dropna()

areas_m = tas_areas.to_crs(28355)  # GDA94 / MGA zone 55 (metres) for buffering
pix_m = pix.to_crs(28355)
rows = []
for buffer_km in (0, 5, 10):
    shapes = areas_m.buffer(buffer_km * 1000)
    hits = gpd.sjoin(pix_m, gpd.GeoDataFrame(geometry=shapes, crs=28355), predicate="within")
    counts = hits.groupby("index_right").size().reindex(range(len(areas_m)), fill_value=0)
    rows.append((buffer_km, int((counts > 0).sum()), int(counts.median())))
for b, n_with, med in rows:
    print(
        f"buffer {b:>2} km: {n_with}/{len(areas_m)} areas have >=1 valid weekly pixel; "
        f"median valid pixels per area = {med}"
    )

buffer  0 km: 28/57 areas have >=1 valid weekly pixel; median valid pixels per area = 0
buffer  5 km: 57/57 areas have >=1 valid weekly pixel; median valid pixels per area = 59
buffer 10 km: 57/57 areas have >=1 valid weekly pixel; median valid pixels per area = 246
